# Battle Lab · Entrenamiento M-C — VGC-Bench sin piedad 🐉⚔️

Libreta canónica de `BATTLE-LAB-MC-TRAIN-001`. El CENSUS_EXTENDED agotó los replays BO3 disponibles sin alcanzar el gate BC-MC, por lo que **LIGHT** parte del checkpoint público BC M-A/M-B y continúa con PPO/self-play M-C. El holdout queda aislado del entrenamiento.


In [ ]:
RUN_MODE = "LIGHT"  # @param ["LIGHT", "NORMAL", "HARD", "CENSUS"]
SYNC_TEAMS = False  # @param {type:"boolean"}
SCRAPE_HUMAN_LOGS = False  # @param {type:"boolean"}
BUILD_TRAJECTORIES = False  # @param {type:"boolean"}
RUN_BC_IF_ENOUGH_DATA = True  # @param {type:"boolean"}
RUN_RL = True  # @param {type:"boolean"}
MIN_BC_TRAJECTORIES = 1000
MIN_BC_TRANSITIONS = 10000
MIN_RATING = 1200
ONLY_WINNER = True
DEVICE = "auto"  # @param ["auto", "cuda", "cpu"]
SEED = 260913
PORT = 8000
PKMN_REPOSITORY = "https://github.com/Iesyo/pkmn.git"
PKMN_REF = "main"
VGC_BENCH_REPOSITORY = "https://github.com/cameronangliss/vgc-bench.git"
VGC_BENCH_COMMIT = "d79f9532947ac114dce1dda2456a590afcd375b2"
NODE_VERSION = "24.21.0"
MODE = {
    "CENSUS": {"max_logs": 5000, "bc_epochs": 0, "rl_steps": 0, "num_envs": 2},
    "LIGHT": {"max_logs": 5000, "bc_epochs": 3, "rl_steps": 196_608, "num_envs": 2},
    "NORMAL": {"max_logs": 20_000, "bc_epochs": 5, "rl_steps": 983_040, "num_envs": 4},
    "HARD": {"max_logs": 50_000, "bc_epochs": 10, "rl_steps": 2_949_120, "num_envs": 4},
}[RUN_MODE]
if RUN_MODE == "CENSUS":
    RUN_RL = False
    RUN_BC_IF_ENOUGH_DATA = False
print("Modo:", RUN_MODE, MODE)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
DRIVE_ROOT = Path("/content/drive/MyDrive/Colabs/LikeNoOneEverWas/BattleLab/MC-Training")
DATA_ROOT = DRIVE_ROOT / "data"
TEAM_DIR = DATA_ROOT / "teams" / "vgcpastes-champions-mc"
SPLIT_ROOT = DATA_ROOT / "team-splits" / f"seed-{SEED}"
TRAIN_TEAM_DIR = SPLIT_ROOT / "train"
HOLDOUT_TEAM_DIR = SPLIT_ROOT / "holdout"
OUTPUT_ROOT = DRIVE_ROOT / "training"
LOG_ROOT = DRIVE_ROOT / "logs"
PKMN_ROOT = Path("/content/pkmn")
VGC_BENCH_ROOT = Path("/content/vgc-bench")
RUNTIME_ROOT = Path("/content/battle-lab-runtime")
SHOWDOWN_ROOT = RUNTIME_ROOT / "pokemon-showdown"
for path in (DATA_ROOT, TEAM_DIR, SPLIT_ROOT, OUTPUT_ROOT, LOG_ROOT):
    path.mkdir(parents=True, exist_ok=True)
print("📁", DRIVE_ROOT)


In [ ]:
import os, subprocess, sys, time

def run(command, *, cwd=None, label=None):
    if label:
        print(f"\n▶ {label}", flush=True)
    subprocess.run([str(x) for x in command], cwd=cwd, check=True)

if not (PKMN_ROOT / ".git").is_dir():
    run(["git", "clone", "--filter=blob:none", PKMN_REPOSITORY, str(PKMN_ROOT)], label="Clonando pkmn")
run(["git", "fetch", "origin", PKMN_REF], cwd=PKMN_ROOT, label="Actualizando pkmn")
run(["git", "checkout", "--force", "FETCH_HEAD"], cwd=PKMN_ROOT, label="Fijando pkmn")
pkmn_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PKMN_ROOT, text=True).strip()
print("pkmn SHA:", pkmn_sha)
run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PKMN_ROOT / "battle_lab" / "requirements-phase1.txt")], label="Dependencias Battle Lab")
run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PKMN_ROOT / "battle_lab" / "requirements-mc-train.txt")], label="Dependencias entrenamiento")
sys.path.insert(0, str(PKMN_ROOT))
from battle_lab.showdown_smoke import DEFAULT_SHOWDOWN_REPOSITORY, ensure_showdown_checkout, install_runtime_config, read_showdown_commit
showdown_commit = read_showdown_commit()
ensure_showdown_checkout(checkout=SHOWDOWN_ROOT, repository=DEFAULT_SHOWDOWN_REPOSITORY, commit=showdown_commit, logs_dir=LOG_ROOT)
install_runtime_config(SHOWDOWN_ROOT)
from battle_lab.vgc_bench_battle import ensure_vgc_bench_checkout
ensure_vgc_bench_checkout(checkout=VGC_BENCH_ROOT, repository=VGC_BENCH_REPOSITORY, commit=VGC_BENCH_COMMIT)
print("Showdown SHA:", showdown_commit)
print("VGC-Bench SHA:", VGC_BENCH_COMMIT)


In [ ]:
import json

def build_mc_base(team_dir):
    return [sys.executable, "-m", "battle_lab.mc_training", "--vgc-bench", str(VGC_BENCH_ROOT), "--project-root", str(PKMN_ROOT), "--data-root", str(DATA_ROOT), "--output-root", str(OUTPUT_ROOT), "--team-dir", str(team_dir), "--showdown", str(SHOWDOWN_ROOT), "--port", str(PORT), "--device", DEVICE, "--seed", str(SEED)]
BASE = build_mc_base(TEAM_DIR)
TRAIN_BASE = build_mc_base(TRAIN_TEAM_DIR)
CENSUS_BASE = [sys.executable, "-m", "battle_lab.mc_census", "--vgc-bench", str(VGC_BENCH_ROOT), "--data-root", str(DATA_ROOT), "--team-dir", str(TEAM_DIR), "--output-root", str(OUTPUT_ROOT)]

def pipeline(base, *args):
    command = base + [str(x) for x in args]
    print("\n$", " ".join(command), flush=True)
    subprocess.run(command, check=True, cwd=PKMN_ROOT)
def mc(*args): pipeline(BASE, *args)
def mc_train(*args): pipeline(TRAIN_BASE, *args)
def census(*args): pipeline(CENSUS_BASE, *args)

if SYNC_TEAMS:
    mc("sync-teams", "--minimum-teams", 150)
else:
    print("⏭️ Reutilizando snapshot de equipos")
run([sys.executable, "-m", "battle_lab.mc_team_split", "--source-dir", str(TEAM_DIR), "--output-root", str(SPLIT_ROOT), "--holdout-fraction", "0.20", "--seed", str(SEED)], cwd=PKMN_ROOT, label="Train/holdout por firma")
split_manifest = json.loads((SPLIT_ROOT / "split_manifest.json").read_text())
if int(split_manifest.get("signatureLeakage", -1)) != 0:
    raise RuntimeError(f"Fuga train/holdout: {split_manifest}")
print(f"✅ train={split_manifest['trainTeams']} · holdout={split_manifest['holdoutTeams']} · fuga=0")


In [ ]:
if SCRAPE_HUMAN_LOGS:
    census("scrape", "--max-logs-per-format", MODE["max_logs"], "--num-workers", 16, "--read-increment", min(20_000, MODE["max_logs"]))
else:
    print("⏭️ Reutilizando battle logs")
logs_manifest_path = DATA_ROOT / "logs_manifest.json"
logs_manifest = json.loads(logs_manifest_path.read_text()) if logs_manifest_path.exists() else {}
accepted_logs = int(logs_manifest.get("totalLogs", 0))
if BUILD_TRAJECTORIES and accepted_logs > 0:
    args = ["build-trajectories", "--num-workers", max(2, (os.cpu_count() or 2) // 2)]
    if MIN_RATING > 0: args += ["--min-rating", MIN_RATING]
    if ONLY_WINNER: args += ["--only-winner"]
    mc(*args)
else:
    print("⏭️ Reutilizando trayectorias")
census("report")


In [ ]:
from battle_lab.showdown_smoke import running_showdown
traj_manifest_path = DATA_ROOT / "trajs_manifest.json"
traj_manifest = json.loads(traj_manifest_path.read_text()) if traj_manifest_path.exists() else {}
traj_count = int(traj_manifest.get("trajectories", 0))
transition_count = int(traj_manifest.get("transitions", 0))
has_bc_data = traj_count >= MIN_BC_TRAJECTORIES and transition_count >= MIN_BC_TRANSITIONS
print(f"Trayectorias={traj_count:,} · transiciones={transition_count:,} · BC habilitable={has_bc_data}")
bc_checkpoint = None
need_server = (RUN_BC_IF_ENOUGH_DATA and has_bc_data and MODE["bc_epochs"] > 0) or (RUN_RL and MODE["rl_steps"] > 0)
if need_server:
    with running_showdown(SHOWDOWN_ROOT, PORT, LOG_ROOT / "mc-training-showdown.log"):
        if RUN_BC_IF_ENOUGH_DATA and has_bc_data and MODE["bc_epochs"] > 0:
            mc_train("bc", "--epochs", MODE["bc_epochs"], "--div-frac", 0.1, "--eval-battles", 50)
            bc_checkpoint = Path(json.loads((OUTPUT_ROOT / "bc" / "summary.json").read_text())["finalCheckpoint"])
        elif RUN_BC_IF_ENOUGH_DATA:
            print("⚠️ Gate BC-MC no alcanzado: self-play partirá del baseline BC M-A/M-B.")
        if RUN_RL and MODE["rl_steps"] > 0:
            rl_args = ["rl", "--total-steps", MODE["rl_steps"], "--num-envs", MODE["num_envs"], "--num-eval-workers", 4]
            if bc_checkpoint is not None: rl_args += ["--initial-checkpoint", str(bc_checkpoint)]
            mc_train(*rl_args)
else:
    print("ℹ️ Modo sin entrenamiento")


In [ ]:
census("report")
summary = json.loads((OUTPUT_ROOT / "census.json").read_text())
rl_summary_path = OUTPUT_ROOT / "rl" / "summary.json"
bc_summary_path = OUTPUT_ROOT / "bc" / "summary.json"
bc_ready = int(summary["trajectories"]) >= MIN_BC_TRAJECTORIES and int(summary["transitions"]) >= MIN_BC_TRANSITIONS
lines = [
    "Battle Lab M-C — latest training run", "",
    f"Modo: {RUN_MODE}",
    f"BC-MC habilitable: {bc_ready}",
    f"Logs humanos M-C: {summary['humanLogs']:,}",
    f"Trayectorias: {summary['trajectories']:,}",
    f"Transiciones: {summary['transitions']:,}",
    f"Train / holdout: {split_manifest['trainTeams']} / {split_manifest['holdoutTeams']}",
    f"Fuga de firmas: {split_manifest['signatureLeakage']}",
    f"pkmn SHA: {pkmn_sha}",
]
if bc_summary_path.exists():
    lines.append(f"BC-MC checkpoint: {json.loads(bc_summary_path.read_text())['finalCheckpoint']}")
else:
    lines.append("BC-MC checkpoint: omitido; baseline BC M-A/M-B")
if rl_summary_path.exists():
    rl = json.loads(rl_summary_path.read_text())
    lines += [f"RL M-C final: {rl['finalCheckpoint']}", f"RL SHA256: {rl['finalCheckpointSha256']}", f"RL segundos: {rl.get('seconds', 'n/a')}"]
report_path = OUTPUT_ROOT / "latest_run.txt"
report_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
print("\n" + "\n".join(lines))
print("📄 Resumen canónico:", report_path)
